# bts-pivot showcase

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/700799/pivot2hist/blob/main/examples/showcase.ipynb)

Auto-fitted pivot tables that flip to histograms and back, with slicing, comparisons,
cell-level explanations, change detection, an LLM prompt export and an interactive
explorer. Built for security logs, but the type guessing is generic, so it works on any
tabular data.

Everything below runs on the bundled sample data, so there's nothing to download. Run
it top to bottom; it takes about a minute.

In [1]:
import importlib.util

if importlib.util.find_spec("bts_pivot") is None:  # Colab or a fresh environment
    %pip install -q "bts-pivot[jupyter]"

import bts_pivot as bp

bp.__version__

'0.21.0'

## 1. Sample data

Synthetic firewall logs: 5,000 connections over a week. `bp.sample.auth_logs()` gives
login events, used further down.

In [2]:
df = bp.sample.firewall_logs()
df.head()

,timestamp,src_ip,dst_ip,dst_port,protocol,action,bytes,duration,severity,rule,country
0,2026-03-01 00:35:16.706486633,10.0.0.23,142.166.3.207,443,TCP,allow,2673,4.608,1,fw-allow-5,IN
1,2026-03-01 00:42:32.887646988,10.0.0.68,76.21.92.87,21,TCP,allow,81709,0.219,5,fw-allow-1,NL
2,2026-03-01 01:29:52.975834628,10.0.0.171,123.126.222.31,53,UDP,allow,270,0.326,5,fw-allow-3,US
3,2026-03-01 01:48:59.474780663,10.0.0.2,169.138.20.114,8080,TCP,allow,2345,2.136,1,fw-allow-1,US
4,2026-03-01 01:57:55.453933602,10.0.0.5,78.169.22.50,80,TCP,allow,536,2.272,1,fw-allow-1,RU


## 2. One call, a fitted pivot

`bp.fit` profiles every column, then searches for the rows, columns and measure that
best fit the box you give it (40 x 12 by default; 15 x 8 here to keep the page short).
It scores candidate layouts on how much the axes tell you about each other, how sparse
the table is and how well it fits the box. The result renders as a heatmap.

In [3]:
v = bp.fit(df, max_rows=15, max_cols=8)
v

pivot · sum(bytes) by dst_port (top 14) x rule · 5,000 rows
rule              fw-allow-3 fw-allow-1 fw-allow-4 fw-allow-5 fw-allow-2 fw-block-2 fw-block-1  \
dst_port (top 14)                                                                                
21                    64,821    119,854     44,066     22,676      8,844         15        788   
22                    56,653    144,384    314,295    231,362     86,502     19,644     21,050   
25                   113,498     81,268     49,623    131,132     93,256      1,667      6,027   
53                   507,381    391,441    660,010    318,624    661,384      1,345      2,383   
80                 1,294,119    863,563    929,486    573,594  1,255,386      4,736      8,926   
123                  134,917     82,732     26,567    161,028    118,450        177        702   
161                  107,069     39,208     55,092    232,229     29,438      1,790        396   
443                1,703,501  1,966,376  1,693,576  1,257,035    847,381      6,589     10,271   
445                  131,709    221,300    141,504    129,556    160,355      5,875      5,067   
1433                  23,983      4,191        779      9,719        233        981        866   
3306                  33,035    112,948     19,419     31,128     32,083        810      5,653   
3389                  44,417    159,289     49,195     45,859    179,480     13,182      2,811   
8080                 220,794    289,773    188,990    101,782    155,530      1,322        375   
8443                 217,473     27,453     77,462    125,410    100,722        286          1   
(other)              224,117    132,494    221,216    117,887    193,351     29,701     19,477   

rule              fw-block-3  
dst_port (top 14)             
21                       442  
22                    12,730  
25                       904  
53                     6,225  
80                     7,880  
123                      633  
161                      133  
443                   13,088  
445                   15,113  
1433                   2,030  
3306                   1,633  
3389                  10,618  
8080                      94  
8443                     172  
(other)               15,351

## 3. It guessed the types

Beyond numeric and categorical, it spots semantic types from the values themselves:
IPv4 addresses, ports, timestamps and their sampling rate. Those drive sensible
bucketing, like treating `dst_port` as a label even though it's a number.

In [4]:
bp.profile(df)

<Profile 5,000 rows x 11 columns>
   column        kind semantic          dtype  nunique  null_frac                                                                     examples
timestamp    datetime    ~1min datetime64[ns]     5000        0.0 2026-03-01 00:35:16.7..., 2026-03-01 00:42:32.8..., 2026-03-01 01:29:52.9...
   src_ip categorical     ipv4            str      288        0.0                                             10.0.0.23, 10.0.0.68, 10.0.0.171
   dst_ip categorical     ipv4            str       80        0.0                                   142.166.3.207, 76.21.92.87, 123.126.222.31
 dst_port categorical     port          int64      335        0.0                                                                  443, 21, 53
 protocol categorical                     str        3        0.0                                                               TCP, UDP, ICMP
   action categorical                     str        3        0.0                                           

## 4. Pivot ⇄ histogram

`toggle()` shows the same layout as a histogram. `histogram()` bins one column, and
picks log bins on its own when the data is heavy-tailed, as `bytes` is here.

In [5]:
v.toggle()

hist · sum(bytes) by dst_port (top 14) x rule · 5,000 rows
dst_port (top 14)  fw-allow-3    fw-allow-1    fw-allow-4    fw-allow-5    fw-allow-2    fw-block-2    fw-block-1    fw-block-3  
21                 ▎      64.8K  ▍       120K  ▏      44.1K  ▏      22.7K  ▏      8.84K  ▏         15  ▏        788  ▏        442
22                 ▏      56.7K  ▌       144K  █       314K  ▊       231K  ▎      86.5K  ▏      19.6K  ▏      21.1K  ▏      12.7K
25                 ▍       113K  ▎      81.3K  ▏      49.6K  ▍       131K  ▎      93.3K  ▏      1.67K  ▏      6.03K  ▏        904
53                 █▌      507K  █▎      391K  ██      660K  █       319K  ██      661K  ▏      1.34K  ▏      2.38K  ▏      6.22K
80                 ████   1.29M  ██▋     864K  ██▉     929K  █▊      574K  ███▉   1.26M  ▏      4.74K  ▏      8.93K  ▏      7.88K
123                ▍       135K  ▎      82.7K  ▏      26.6K  ▌       161K  ▍       118K  ▏        177  ▏        702  ▏        633
161                ▍       107K  ▏      39.2K  ▏      55.1K  ▊       232K  ▏      29.4K  ▏      1.79K  ▏        396  ▏        133
443                █████▎  1.7M  ██████ 1.97M  █████▏ 1.69M  ███▉   1.26M  ██▋     847K  ▏      6.59K  ▏      10.3K  ▏      13.1K
445                ▍       132K  ▋       221K  ▍       142K  ▍       130K  ▌       160K  ▏      5.88K  ▏      5.07K  ▏      15.1K
1433               ▏        24K  ▏      4.19K  ▏        779  ▏      9.72K  ▏        233  ▏        981  ▏        866  ▏      2.03K
3306               ▏        33K  ▍       113K  ▏      19.4K  ▏      31.1K  ▏      32.1K  ▏        810  ▏      5.65K  ▏      1.63K
3389               ▏      44.4K  ▌       159K  ▏      49.2K  ▏      45.9K  ▌       179K  ▏      13.2K  ▏      2.81K  ▏      10.6K
8080               ▋       221K  ▉       290K  ▋       189K  ▎       102K  ▌       156K  ▏      1.32K  ▏        375  ▏         94
8443               ▋       217K  ▏      27.5K  ▎      77.5K  ▍       125K  ▎       101K  ▏        286  ▏          1  ▏        172
(other)            ▋       224K  ▍       132K  ▋       221K  ▍       118K  ▋       193K  ▏      29.7K  ▏      19.5K  ▏      15.4K

In [6]:
v.histogram("bytes", by="action")

hist · count by bytes (bins) x action · 5,000 rows
bytes (bins)  allow                       deny                        drop                      
[0, 1)                                 0  ▌                       16  ▏                        4
[1, 2)        ▏                        1  ▍                       14  ▎                        8
[2, 5)        ▏                        5  █                       33  ▋                       20
[5, 10)       ▍                       12  █▊                      55  ▊                       26
[10, 20)      █▏                      35  ██▊                     89  ▉                       30
[20, 50)      ███▌                   111  ███▉                   125  █▋                      54
[50, 100)     █████▋                 181  ███▍                   107  █▍                      45
[100, 200)    ██████████▍            333  ███▋                   115  █                       33
[200, 500)    █████████████████████▍ 688  ██▉                     91  █                       34
[500, 1K)     ██████████████████▏    583  █▎                      42  ▌                       17
[1K, 2K)      █████████████████▉     574  ▋                       22  ▍                       11
[2K, 5K)      ██████████████████████ 707  ▎                       10  ▏                        5
[5K, 10K)     ██████████▎            330  ▏                        4                           0
[10K, 20K)    ██████▋                212  ▏                        1                           0
[20K, 50K)    ████▊                  153                           0                           0
[50K, 100K)   █▎                      41                           0                           0
[100K, 200K)  ▌                       17                           0                           0
[200K, 500K)  ▏                        4                           0                           0
[500K, 1M]    ▏                        2                           0                           0

## 5. Slicing

Slices stack, show in the title, and survive toggling. They accept plain values, lists,
comparisons, percentiles and relative time.

In [7]:
v.slice(action="deny", timestamp="last 24h")

pivot · sum(bytes) by dst_port (top 14) x rule · 95 rows of 5,000 · slices: action=deny & timestamp=last 24h
rule              fw-block-2 fw-block-3 fw-block-1
dst_port (top 14)                                 
21                         0          0        118
22                     2,821      6,535      1,703
25                     1,465          0         45
53                         0        130         54
80                       241        133        133
443                      111      1,410      2,845
445                      587        751        871
1433                       0        100        778
3306                      85        863        117
3389                     165        217          0
5900                       0        569          0
14966                    208          0          0
34345                    268          0          0
58750                      0         42          0
(other)                1,781      1,097        364

In [8]:
v.slice(bytes=">= p95").top("dst_ip", 5)

pivot · sum(bytes) by dst_port (top 14) x rule · 116 rows of 5,000 · slices: bytes>=p95 (17.9K) & dst_ip∈top5
rule              fw-allow-3 fw-allow-4 fw-allow-1 fw-allow-2 fw-allow-5
dst_port (top 14)                                                       
21                         0          0     81,709          0          0
22                         0    190,166     42,689     19,421     19,325
25                         0          0     30,322     30,556          0
53                   178,790    331,534     19,514    207,607     56,379
80                   280,739    178,391    203,096    195,293    100,317
123                        0          0     28,836     34,198          0
161                   42,280     26,467          0          0    121,002
443                  911,840    369,713  1,073,334    164,274    437,302
445                   21,546     18,910     53,316     18,127          0
3306                  21,100          0     31,696     22,959          0
3389                       0          0    134,342    121,367          0
8080                  86,234          0     67,990          0          0
8443                 148,363          0          0          0    101,911
61147                 34,396          0          0          0          0
(other)               60,362     38,537     98,901    126,897          0

## 6. Drill along a hierarchy

Semantic types come with levels. `coarser()` rolls source IPs up from hosts to /24
networks; `finer()` goes back down.

In [9]:
ips = bp.fit(df, rows=["src_ip"], cols=["action"])
ips.coarser("src_ip")

pivot · sum(bytes) by src_ip (/24) x action · 5,000 rows
action             allow     deny    drop
src_ip (/24)                             
10.0.0.0/24   21,053,330  186,067  66,669
10.0.1.0/24      343,127    6,358     865

## 7. Not sure what to look at?

`suggest()` returns the next-best layouts from the same search, best first, and
`use(i)` switches to one.

In [10]:
for i, layout in enumerate(v.suggest()):
    print(i, layout)

0 <Layout sum(bytes) by dst_port (top 14) x rule>
1 <Layout sum(bytes) by dst_port (top 14) x timestamp (day)>
2 <Layout sum(bytes) by dst_port (top 14) x timestamp (weekday)>
3 <Layout sum(bytes) by dst_ip (top 14) x rule>
4 <Layout sum(bytes) by dst_ip (top 14) x timestamp (day)>


In [11]:
v.use(2)

pivot · sum(bytes) by dst_port (top 14) x timestamp (weekday) · 5,000 rows
timestamp (weekday)        Mon      Tue      Wed        Thu        Fri        Sat      Sun
dst_port (top 14)                                                                         
21                       6,361   47,209   12,471     14,069     13,017     28,536  139,843
22                     257,394   67,886  178,155     92,207     83,509    134,715   72,754
25                     140,121   76,697   13,634     25,489     92,061     88,292   41,081
53                     428,670  290,164  508,615    435,092    229,884    384,193  272,175
80                   1,127,718  539,189  826,599    536,592    579,371    617,886  710,335
123                     72,699   55,952   30,784    197,151     49,955     63,236   55,429
161                     78,792  144,352   79,430     19,753     24,361     33,505   85,162
443                  1,578,628  567,285  622,058  1,536,384  1,434,845  1,234,492  524,125
445                    177,848  117,325  162,852    117,392     60,458     80,086   94,518
1433                     5,583   17,287    5,235      4,574      1,897      1,975    6,231
3306                    34,079   72,375   48,886     21,629     10,503      9,099   40,138
3389                   166,457  171,235   31,695     29,933     32,262     40,468   32,801
8080                   111,866  112,600  210,081    104,512    127,950    137,097  154,554
8443                   113,879  211,372   98,664     26,155     15,898     48,203   34,808
(other)                190,725  196,196  185,785     45,184     70,854    185,367   79,483

## 8. What's surprising

`heat="surprise"` colours each cell by how far it sits from what its row and column
totals predict, rather than by raw size, so a big-but-expected cell stays pale.
`anomalies()` lists the same cells, most surprising first.

In [12]:
v.style(heat="surprise")

pivot · sum(bytes) by dst_port (top 14) x rule · 5,000 rows
rule              fw-allow-3 fw-allow-1 fw-allow-4 fw-allow-5 fw-allow-2 fw-block-2 fw-block-1  \
dst_port (top 14)                                                                                
21                    64,821    119,854     44,066     22,676      8,844         15        788   
22                    56,653    144,384    314,295    231,362     86,502     19,644     21,050   
25                   113,498     81,268     49,623    131,132     93,256      1,667      6,027   
53                   507,381    391,441    660,010    318,624    661,384      1,345      2,383   
80                 1,294,119    863,563    929,486    573,594  1,255,386      4,736      8,926   
123                  134,917     82,732     26,567    161,028    118,450        177        702   
161                  107,069     39,208     55,092    232,229     29,438      1,790        396   
443                1,703,501  1,966,376  1,693,576  1,257,035    847,381      6,589     10,271   
445                  131,709    221,300    141,504    129,556    160,355      5,875      5,067   
1433                  23,983      4,191        779      9,719        233        981        866   
3306                  33,035    112,948     19,419     31,128     32,083        810      5,653   
3389                  44,417    159,289     49,195     45,859    179,480     13,182      2,811   
8080                 220,794    289,773    188,990    101,782    155,530      1,322        375   
8443                 217,473     27,453     77,462    125,410    100,722        286          1   
(other)              224,117    132,494    221,216    117,887    193,351     29,701     19,477   

rule              fw-block-3  
dst_port (top 14)             
21                       442  
22                    12,730  
25                       904  
53                     6,225  
80                     7,880  
123                      633  
161                      133  
443                   13,088  
445                   15,113  
1433                   2,030  
3306                   1,633  
3389                  10,618  
8080                      94  
8443                     172  
(other)               15,351

In [13]:
v.anomalies(5)

,row,col,observed,expected,residual,direction
0,161,fw-allow-5,232229.0,74972.395,574.326,over
1,443,fw-allow-2,847381.0,1357999.399,-438.174,under
2,(other),fw-block-2,29701.0,3880.176,414.519,over
3,80,fw-allow-2,1255386.0,894310.978,381.815,over
4,22,fw-allow-3,56653.0,199685.743,-320.083,under


## 9. Insights, computed locally

A deterministic statistical pass over whatever is in view: skew, multi-modal columns,
concentration, outliers, correlated columns, surprising cells and spikes. No model is
called.

In [14]:
for f in v.insights()["findings"][:8]:
    print(f"[{f['kind']}] {f['text']}")

[skew] `bytes` is strongly right-skewed (skewness 20.27) - a few very large values pull the mean up; median is a better summary than mean here.
[anomaly] 161 / fw-allow-5 is over expectation: observed 2.322e+05 vs. expected 7.497e+04 (residual +574.3).
[outliers] `bytes` has 12.7% of values outside 1.5x the interquartile range (outside [-3.54e+03, 6.3e+03]).
[correlation] `action` and `rule` move together (normalized mutual information 0.80).
[correlation] `dst_port` and `protocol` move together (normalized mutual information 0.78).
[anomaly] 443 / fw-allow-2 is under expectation: observed 8.474e+05 vs. expected 1.358e+06 (residual -438.2).
[skew] `duration` is strongly right-skewed (skewness 2.21) - a few very large values pull the mean up; median is a better summary than mean here.
[anomaly] (other) / fw-block-2 is over expectation: observed 2.97e+04 vs. expected 3,880 (residual +414.5).


## 10. Compare A vs B on one layout

Denied traffic against everything else, on one shared set of rows, columns and bins, so
the two sides are actually comparable. Cells show lift; `top()` lists the movers and
`drivers()` explains what else differs between the two sides.

In [15]:
ports = bp.fit(df, rows=["dst_port"], cols=["action"], max_rows=12)
cmp = ports.compare(action="deny")
cmp

compare · sum(bytes) by dst_port (top 11) x rule · action=deny (724 rows) vs rest (4,276 rows) · lift
rule              fw-allow-3 fw-allow-1 fw-allow-4 fw-allow-5 fw-allow-2 fw-block-2 fw-block-1  \
dst_port (top 11)                                                                                
21                        ×0         ×0         ×0         ×0         ×0         ×0        ×29   
22                        ×0         ×0         ×0         ×0         ×0     ×1,285       ×359   
25                        ×0         ×0         ×0         ×0         ×0     ×1,438     ×1,889   
53                        ×0         ×0         ×0         ×0         ×0        ×84       ×477   
80                        ×0         ×0         ×0         ×0         ×0       ×198        ×33   
123                       ×0         ×0         ×0         ×0         ×0        new       ×460   
161                       ×0         ×0         ×0         ×0         ×0       ×665    ×10,931   
443                       ×0         ×0         ×0         ×0         ×0        ×22       ×396   
445                       ×0         ×0         ×0         ×0         ×0       ×248       ×336   
3389                      ×0         ×0         ×0         ×0         ×0       ×112     ×1,217   
8080                      ×0         ×0         ×0         ×0         ×0    ×36,754       ×213   
(other)                   ×0         ×0         ×0         ×0         ×0     ×1,526       ×225   

rule              fw-block-3  
dst_port (top 11)             
21                    ×4,371  
22                    ×1,432  
25                   ×50,307  
53                      ×231  
80                      ×278  
123                      new  
161                     ×5.3  
443                   ×1,080  
445                     ×166  
3389                     ×74  
8080                    ×138  
(other)                 ×942  

top 5 by |lift| (A's share of its own total ÷ B's share of its total (>1: over-represented in A)):
    row        col action=deny      rest      delta ratio   lift
(other) fw-block-2      29,614     2,164    +27,450   ×14 ×1,526
     22 fw-block-2      18,075     1,569    +16,506   ×12 ×1,285
    443 fw-allow-1           0 1,966,376 -1,966,376    ×0     ×0
(other) fw-block-3      17,154     2,032    +15,122  ×8.4   ×942
(other) fw-block-1      17,384     8,613     +8,771    ×2   ×225

In [16]:
for line in cmp.drivers().head(5)["text"]:
    print(line)

`bytes` median 62.5 in action=deny vs 924 in rest (×0.068)
`duration` median 0.16 in action=deny vs 1.58 in rest (×0.1)
`severity` is '1' for 0% of action=deny vs 43% of rest (×0.012)
`severity` is '2' for 46% of action=deny vs 29% of rest (×1.6)
`protocol` is 'UDP' for 5% of action=deny vs 18% of rest (×0.32)


## 11. Interrogate one cell

Name a cell by the labels the table shows. `explain()` says how it compares with what
independence predicts, its shares of row, column and table, and what sets its rows
apart. `rows()` returns the raw rows behind it.

In [17]:
ports.explain("22", "deny")

dst_port=22 × action=deny: sum(bytes) = 45,942 · ×5.8 what independence of the axes predicts (7,877.94) - over · 5% of its row, 24% of its column, 0.2% of the table · rank 14 of 36 cells · 144 of 5,000 rows
What sets these rows apart: `bytes` median 52 here vs 685 elsewhere (×0.076); `duration` median 0.151 here vs 1.24 elsewhere (×0.12); `rule` is 'fw-block-2' for 42% of these rows vs 6% elsewhere (×6.8); `rule` is 'fw-block-1' for 38% of these rows vs 6% elsewhere (×6.1); `severity` is '2' for 53% of these rows vs 31% elsewhere (×1.7).

In [18]:
ports.rows("22", "deny").head()

,timestamp,src_ip,dst_ip,dst_port,protocol,action,bytes,duration,severity,rule,country
22,2026-03-01 04:27:24.298171960,10.0.0.10,222.185.151.94,22,TCP,deny,16,0.085,2,fw-block-3,US
32,2026-03-01 05:13:37.671521028,10.0.0.3,138.173.230.6,22,TCP,deny,14,0.900,3,fw-block-2,CN
53,2026-03-01 06:33:04.911062242,10.0.0.24,32.24.11.167,22,TCP,deny,414,0.686,2,fw-block-1,CN
54,2026-03-01 06:34:26.062029674,10.0.0.1,31.104.77.226,22,TCP,deny,31,0.073,2,fw-block-2,US
118,2026-03-01 08:31:44.637348695,10.0.0.11,194.194.216.167,22,TCP,deny,217,0.352,4,fw-block-1,US


## 12. What changed, and what's new

`spikes()` scores each row's trend over time against its own seasonal baseline and
lists spikes, drops and step changes. `novel()` lists what appeared recently per entity:
new pairs, unseen values, fan-out jumps.

In [19]:
ports.spikes("timestamp")[["row", "bucket", "kind", "observed", "expected", "score"]]

,row,bucket,kind,observed,expected,score
0,445,2026-03-06 12:00,shift down,103941.0,165136.7151,-3.84


In [20]:
for line in v.novel("src_ip", "dst_port").head(5)["text"]:
    print(line)

src_ip 10.0.0.60 → dst_port 443 is a new pair: not before 2026-03-06 06:09; 443 had been seen with 177 of 280 src_ips, 3 row(s)
src_ip 10.0.0.120 fans out: 3 distinct dst_port since 2026-03-06 06:09 vs 1 before (×3)
src_ip 10.0.0.196 fans out: 3 distinct dst_port since 2026-03-06 06:09 vs 1 before (×3)


## 13. Sequences: Markov chains

On the auth logs: how often each event follows another, per user in time order, as a
pivot of `from` x `to`, plus the most common three-step chains.

In [21]:
auth = bp.sample.auth_logs()
bp.chains(auth, "event", by="user", time="timestamp")

pivot · sum(count) by from x to · 9 rows
to            login_success logout login_failure
from                                            
login_failure           257     71            51
login_success         1,235    404           247
logout                  400    133            82

In [22]:
bp.sequences(auth, "event", by="user", time="timestamp", length=3).head()

,chain,count,share,groups
0,login_success → login_success → login_success,787,0.285042,89
1,login_success → logout → login_success,262,0.094893,71
2,login_success → login_success → logout,249,0.090185,74
3,logout → login_success → login_success,238,0.086201,75
4,login_failure → login_success → login_success,161,0.058312,60


## 14. Themes and trend sparklines

Three themes (`light`, `graphite`, `gunmetal`). `sparklines=` adds a trend column: one
small line per row, that row's measure over time.

In [23]:
ports.style(theme="graphite", sparklines="timestamp")

pivot · sum(bytes) by dst_port (top 11) x action · 5,000 rows
action                 allow    deny    drop
dst_port (top 11)                           
21                   260,261     594     651
22                   833,196  45,942   7,482
25                   468,777   8,140     458
53                 2,538,840   6,704   3,249
80                 4,916,148  10,708  10,834
123                  523,694   1,375     137
161                  463,036   1,931     388
443                7,467,869  20,942   9,006
445                  784,424  16,902   9,153
3389                 478,240  13,419  13,192
8080                 956,869   1,616     175
(other)            1,705,103  64,152  12,809

## 15. Big files: survey first, then page

Pass a path and it surveys the file against your free memory before loading anything.
If the file won't fit the budget, it fits the layout on a sample and aggregates page by
page; counts and sums stay exact. Here a tiny 3 MB budget forces that on a 50,000-row
CSV (left to itself, the budget is a share of your free memory).

In [24]:
import os, tempfile

path = os.path.join(tempfile.mkdtemp(), "firewall_50k.csv")
bp.sample.firewall_logs(50_000).to_csv(path, index=False)
bp.survey(path, memory_budget_mb=3)

<Survey /tmp/tmp2zuyhttk/firewall_50k.csv>
csv: 4.6 MB on disk, 49,995 (est.) rows x 11 columns, ~8 MB in memory (158 B/row)
machine: 15,182 MB free of 16,095 MB; 4 CPUs, load 0.57, this process 190 MB
budget: 3 MB -> choke; plan: paged: 3 pages of 20,000 rows, fit on a 5,696-row sample; ~8 MB exceeds the 3 MB budget

In [25]:
big = bp.fit(path, memory_budget_mb=3, max_rows=10, max_cols=6)
big

pivot · sum(bytes) by dst_port (top 9) x rule (top 5) · ≈49,995 rows (paged, 3 pages)
rule (top 5)      fw-allow-4  fw-allow-3  fw-allow-5  fw-allow-1  fw-allow-2  (other)
dst_port (top 9)                                                                     
22                 1,753,125   2,957,482   2,364,379   1,960,451   1,621,493  587,052
25                 1,037,796   1,114,896   1,438,923   1,209,266     982,298   24,115
53                 4,442,003   4,972,708   6,704,829   5,216,453   6,102,445  141,346
80                 8,559,737   9,670,021   9,957,243   9,455,627   8,741,936  156,862
123                1,124,321   1,523,551   2,018,313   1,301,730   1,389,867   19,853
443               16,681,735  13,006,930  14,067,511  14,444,305  17,480,641  388,350
445                1,327,120   1,485,735   1,062,375   1,193,684   1,104,302  366,063
3389                 854,893   1,053,096     816,855     711,584   2,452,828  293,807
8080               2,077,796   2,161,327   2,207,880   2,128,001   2,323,406   57,925
(other)            6,818,469   4,574,388   5,238,631   6,561,023   6,409,184  775,727

## 16. Hand it to an LLM

`llm_context()` packages a view as a short description, metadata and a markdown table
sized for a model's context. `prompt()` builds one self-contained prompt with the
dataset, table, surprising cells and your question. Nothing here calls a model; the
same functions are exposed to agents through the `bts-pivot-mcp` server.

In [26]:
print(v.llm_context()["description"])

Pivot table of 5,000 rows: sum(bytes) by dst_port (top 14) x rule. Shown as 15 row(s) x 8 column(s). Most surprising cell: 161 / fw-allow-5 (over expectation - observed 2.322e+05 vs. expected 7.497e+04).


In [27]:
p = ports.prompt("Which ports and rules look like an attack?")
print(f"~{p.tokens:,} tokens; the first part:\n")
print("\n".join(p.splitlines()[:38]))
print("\n… then the surprising cells, findings and your question")

~558 tokens; the first part:

# Data analysis request

The facts below were computed locally by bts-pivot from the actual data; nothing here is invented or sampled from a model. Reason only from these facts, and quote specific cells and numbers when you do.

## Dataset

5,000 rows in view, 11 columns.

- `timestamp`: datetime, ~every 1min; 5,000 distinct; e.g. 2026-03-01 00:35:16.7..., 2026-03-01 00:42:32.8..., 2026-03-01 01:29:52.9...
- `src_ip`: categorical, ipv4; 288 distinct; e.g. 10.0.0.23, 10.0.0.68, 10.0.0.171
- `dst_ip`: categorical, ipv4; 80 distinct; e.g. 142.166.3.207, 76.21.92.87, 123.126.222.31
- `dst_port`: categorical, port; 335 distinct; e.g. 443, 21, 53
- `protocol`: categorical; 3 distinct; e.g. TCP, UDP, ICMP
- `action`: categorical; 3 distinct; e.g. allow, deny, drop
- `bytes`: numeric; 2,629 distinct; e.g. 2673, 81709, 270
- `duration`: numeric; 2,963 distinct; e.g. 4.608, 0.219, 0.326
- `severity`: categorical; 5 distinct; e.g. 1, 5, 4
- `rule`: categorical; 8 dis

## 17. The interactive explorer

The two cells below need a live notebook (Jupyter, VS Code or Colab), since widgets
don't render on a static page.

`bp.explore(df)` opens the full explorer: drag fields into rows, columns, values and
slicers; flip between pivot, histogram and chains; click any cell to explain it; compare,
style, and copy the code that reproduces the view.

![explorer, Fields tab, graphite theme](https://raw.githubusercontent.com/700799/pivot2hist/main/docs/explorer-fields-graphite.png)

In [ ]:
bp.explore(df)

`bp.explore()` with no data opens a start screen: upload files or a whole folder,
type the path of a file or folder (in Colab, a mounted Drive path), or open a demo.

![explorer start screen](https://raw.githubusercontent.com/700799/pivot2hist/main/docs/explorer-start.png)

In [ ]:
bp.explore()

## More

- The README covers every option: https://github.com/700799/pivot2hist
- Shell: `bts-pivot firewall.csv --slice action=deny --hist --on bytes --by dst_port`,
  or `bts-pivot --demo firewall`
- For LLM agents: `bts-pivot-mcp`, an MCP server with the same operations